# 🧠 Mini LLM + Chatbot from Scratch

**Framework:** CRISP-DM  
**Model:** decoder-only Transformer  
**Goal:** train a small educational LLM that can fit on a laptop GPU and use it through a simple chatbot loop.

### Modern primitives included
- RoPE rotary positional embeddings
- RMSNorm
- SwiGLU feed-forward network
- Grouped-Query Attention (GQA)
- PyTorch scaled-dot-product attention
- residual pre-normalization
- tied token embedding / LM head
- AdamW + cosine learning-rate schedule
- gradient clipping
- mixed precision when CUDA is available
- top-k + nucleus/top-p sampling
- repetition penalty

The default model is intentionally small enough to inspect and train locally.


## CRISP-DM 1 — Business Understanding

### Objective
Build a compact generative language model for educational chatbot experiments without requiring a datacenter GPU.

### Success criteria
- model trains without out-of-memory errors on a typical laptop GPU;
- validation loss decreases;
- generated text becomes more coherent than random output;
- chatbot inference works with controllable sampling;
- architecture uses modern LLM design ideas while remaining understandable.

### Scope
This is an **educational small language model**, not a replacement for a production-scale assistant. Small models trained on TinyStories are specialized toward simple story-like language and have limited world knowledge.


In [ ]:
# Optional dependencies for the full TinyStories run.
import importlib.util, subprocess, sys

packages = {
    "datasets":"datasets",
    "tokenizers":"tokenizers"
}
for module, package in packages.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",package])

print("Dependencies ready.")


In [ ]:
import math, random, os, time
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:",device)
if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0))


## CRISP-DM 2 — Data Understanding

We use **TinyStories**, a dataset created specifically to study how small language models learn coherent English.

For laptop training, the notebook streams only a configurable number of stories rather than downloading the entire multi-million-story corpus.

Recommended starting point:
- 5,000–20,000 stories for a quick experiment;
- increase gradually if your GPU and training time allow.


In [ ]:
MAX_STORIES = 8000

stream = load_dataset("roneneldan/TinyStories", split="train", streaming=True)

stories = []
for row in stream:
    text = row.get("text","").strip()
    if text:
        stories.append(text)
    if len(stories) >= MAX_STORIES:
        break

print("Stories loaded:",len(stories))
print("Example:")
print(stories[0][:500])


## CRISP-DM 3 — Data Preparation

### Tokenizer
Train a small byte-level BPE tokenizer on the selected story subset.

### Sequence construction
The stories are concatenated with an end-of-text marker and converted into fixed-length causal language-model sequences.

This keeps preprocessing simple and avoids padding-heavy batches.


In [ ]:
VOCAB_SIZE = 4096
SPECIAL = ["<pad>","<bos>","<eos>","<unk>"]

tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=SPECIAL,
    min_frequency=2
)
tokenizer.train_from_iterator(stories, trainer=trainer)

print("Tokenizer vocab:",tokenizer.get_vocab_size())


In [ ]:
EOS_ID = tokenizer.token_to_id("<eos>")
all_ids = []

for text in stories:
    all_ids.extend(tokenizer.encode(text).ids)
    all_ids.append(EOS_ID)

tokens = torch.tensor(all_ids,dtype=torch.long)

split = int(len(tokens)*.95)
train_tokens = tokens[:split]
val_tokens = tokens[split:]

print("Total tokens:",len(tokens))
print("Train tokens:",len(train_tokens))
print("Validation tokens:",len(val_tokens))


## CRISP-DM 4 — Modeling

The model is a modern decoder-only Transformer.

### Why these primitives?

**RMSNorm** normalizes activations with less computation than LayerNorm.

**RoPE** injects relative position information directly into query/key vectors.

**Grouped-Query Attention** keeps several query heads but shares fewer key/value heads, reducing KV memory.

**SwiGLU** uses gated nonlinear feed-forward computation.

**Weight tying** shares token-embedding and output projection weights, reducing parameters.


In [ ]:
@dataclass
class Config:
    vocab_size: int = VOCAB_SIZE
    dim: int = 256
    n_layers: int = 6
    n_heads: int = 8
    n_kv_heads: int = 2
    hidden_dim: int = 768
    max_seq_len: int = 256
    dropout: float = 0.0

cfg = Config()


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self,dim,eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self,x):
        return self.weight * x * torch.rsqrt(x.pow(2).mean(-1,keepdim=True)+self.eps)

def precompute_rope(head_dim,seq_len,base=10000.0):
    inv = 1.0/(base**(torch.arange(0,head_dim,2).float()/head_dim))
    pos = torch.arange(seq_len).float()
    freq = torch.outer(pos,inv)
    return torch.cos(freq),torch.sin(freq)

def apply_rope(x,cos,sin):
    x1,x2=x[...,::2],x[...,1::2]
    c=cos[:x.size(-2)].to(x.device)[None,None,:,:]
    s=sin[:x.size(-2)].to(x.device)[None,None,:,:]
    return torch.stack((x1*c-x2*s,x1*s+x2*c),dim=-1).flatten(-2)


In [ ]:
class GQAttention(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.n_heads=cfg.n_heads
        self.n_kv_heads=cfg.n_kv_heads
        self.head_dim=cfg.dim//cfg.n_heads

        self.q_proj=nn.Linear(cfg.dim,cfg.n_heads*self.head_dim,bias=False)
        self.k_proj=nn.Linear(cfg.dim,cfg.n_kv_heads*self.head_dim,bias=False)
        self.v_proj=nn.Linear(cfg.dim,cfg.n_kv_heads*self.head_dim,bias=False)
        self.o_proj=nn.Linear(cfg.n_heads*self.head_dim,cfg.dim,bias=False)

        cos,sin=precompute_rope(self.head_dim,cfg.max_seq_len)
        self.register_buffer("rope_cos",cos,persistent=False)
        self.register_buffer("rope_sin",sin,persistent=False)

    def forward(self,x):
        B,T,C=x.shape
        q=self.q_proj(x).view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        k=self.k_proj(x).view(B,T,self.n_kv_heads,self.head_dim).transpose(1,2)
        v=self.v_proj(x).view(B,T,self.n_kv_heads,self.head_dim).transpose(1,2)

        q=apply_rope(q,self.rope_cos,self.rope_sin)
        k=apply_rope(k,self.rope_cos,self.rope_sin)

        repeat=self.n_heads//self.n_kv_heads
        k=k.repeat_interleave(repeat,dim=1)
        v=v.repeat_interleave(repeat,dim=1)

        y=F.scaled_dot_product_attention(q,k,v,is_causal=True)
        y=y.transpose(1,2).contiguous().view(B,T,C)
        return self.o_proj(y)

class SwiGLU(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.w1=nn.Linear(cfg.dim,cfg.hidden_dim,bias=False)
        self.w2=nn.Linear(cfg.hidden_dim,cfg.dim,bias=False)
        self.w3=nn.Linear(cfg.dim,cfg.hidden_dim,bias=False)
    def forward(self,x):
        return self.w2(F.silu(self.w1(x))*self.w3(x))


In [ ]:
class Block(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.attn_norm=RMSNorm(cfg.dim)
        self.ffn_norm=RMSNorm(cfg.dim)
        self.attn=GQAttention(cfg)
        self.ffn=SwiGLU(cfg)
    def forward(self,x):
        x=x+self.attn(self.attn_norm(x))
        x=x+self.ffn(self.ffn_norm(x))
        return x

class MiniLLM(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.cfg=cfg
        self.embed=nn.Embedding(cfg.vocab_size,cfg.dim)
        self.blocks=nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm=RMSNorm(cfg.dim)
        self.lm_head=nn.Linear(cfg.dim,cfg.vocab_size,bias=False)

        # Small GPT-style initialization keeps initial logits well scaled.
        self.apply(self._init_weights)

        # Weight tying reduces parameters and improves token representation sharing.
        self.lm_head.weight=self.embed.weight

    def _init_weights(self,module):
        if isinstance(module,nn.Linear):
            nn.init.normal_(module.weight,mean=0.0,std=0.02)
        elif isinstance(module,nn.Embedding):
            nn.init.normal_(module.weight,mean=0.0,std=0.02)

    def forward(self,idx,targets=None):
        x=self.embed(idx)
        for block in self.blocks:
            x=block(x)
        logits=self.lm_head(self.norm(x))
        loss=None
        if targets is not None:
            loss=F.cross_entropy(
                logits.reshape(-1,logits.size(-1)),
                targets.reshape(-1)
            )
        return logits,loss

model=MiniLLM(cfg).to(device)
params=sum(p.numel() for p in model.parameters())
print(f"Parameters: {params/1e6:.2f}M")
print(f"FP16 model weights: {params*2/1024**2:.1f} MB")


### Laptop presets

If training is too slow or you run out of memory:

```python
# Smaller
dim=192, n_layers=4, n_heads=6, n_kv_heads=2, hidden_dim=576

# Default
dim=256, n_layers=6, n_heads=8, n_kv_heads=2, hidden_dim=768

# Larger laptop GPU
dim=384, n_layers=8, n_heads=8, n_kv_heads=2, hidden_dim=1152
```

Reduce `BATCH_SIZE` before reducing model quality.


In [ ]:
BATCH_SIZE=16 if device=="cuda" else 4
SEQ_LEN=cfg.max_seq_len
TRAIN_STEPS=1200
EVAL_EVERY=100
EVAL_BATCHES=20
GRAD_ACCUM=2
MAX_LR=3e-4
MIN_LR=3e-5
WARMUP_STEPS=100

def get_batch(data):
    ix=torch.randint(0,len(data)-SEQ_LEN-1,(BATCH_SIZE,))
    x=torch.stack([data[i:i+SEQ_LEN] for i in ix])
    y=torch.stack([data[i+1:i+SEQ_LEN+1] for i in ix])
    return x.to(device),y.to(device)

def lr_for_step(step):
    if step < WARMUP_STEPS:
        return MAX_LR*(step+1)/WARMUP_STEPS
    ratio=(step-WARMUP_STEPS)/max(1,TRAIN_STEPS-WARMUP_STEPS)
    coeff=.5*(1+math.cos(math.pi*ratio))
    return MIN_LR+coeff*(MAX_LR-MIN_LR)

optimizer=torch.optim.AdamW(
    model.parameters(),
    lr=MAX_LR,
    betas=(.9,.95),
    weight_decay=.1
)

use_amp = device=="cuda"
scaler = torch.amp.GradScaler("cuda",enabled=use_amp)


In [ ]:
@torch.no_grad()
def estimate_loss(data,batches=EVAL_BATCHES):
    model.eval()
    vals=[]
    for _ in range(batches):
        x,y=get_batch(data)
        with torch.autocast(device_type=device,dtype=torch.float16,enabled=use_amp):
            _,loss=model(x,y)
        vals.append(loss.item())
    model.train()
    return float(np.mean(vals))

history=[]

for step in range(TRAIN_STEPS):
    optimizer.zero_grad(set_to_none=True)
    total=0.0

    for _ in range(GRAD_ACCUM):
        x,y=get_batch(train_tokens)
        with torch.autocast(device_type=device,dtype=torch.float16,enabled=use_amp):
            _,loss=model(x,y)
            loss=loss/GRAD_ACCUM
        scaler.scale(loss).backward()
        total += loss.item()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)

    lr=lr_for_step(step)
    for group in optimizer.param_groups:
        group["lr"]=lr

    scaler.step(optimizer)
    scaler.update()

    if step % EVAL_EVERY == 0 or step == TRAIN_STEPS-1:
        val_loss=estimate_loss(val_tokens)
        ppl=math.exp(min(val_loss,20))
        history.append([step,total,val_loss,ppl,lr])
        print(f"step {step:4d} | train {total:.3f} | val {val_loss:.3f} | ppl {ppl:.1f}")

history_df=pd.DataFrame(
    history,
    columns=["step","train_loss","val_loss","perplexity","lr"]
)
display(history_df)


## CRISP-DM 5 — Evaluation

For a small generative model, useful checks include:

- validation cross-entropy;
- perplexity;
- sample coherence;
- repetition;
- whether output stays within the learned domain;
- inference speed and GPU memory.

Perplexity is useful but not sufficient: generation samples still need qualitative inspection.


In [ ]:
plt.figure(figsize=(10,5))
plt.plot(history_df["step"],history_df["val_loss"],marker="o")
plt.xlabel("Training step")
plt.ylabel("Validation loss")
plt.title("Validation Loss")
plt.tight_layout()
plt.show()


## Text Generation

Use temperature, top-k, and nucleus/top-p sampling rather than always selecting the highest-probability token.


In [ ]:
@torch.no_grad()
def generate(
    model,
    idx,
    max_new_tokens=120,
    temperature=.8,
    top_k=50,
    top_p=.95,
    repetition_penalty=1.08
):
    model.eval()

    for _ in range(max_new_tokens):
        x=idx[:,-model.cfg.max_seq_len:]
        logits,_=model(x)
        logits=logits[:,-1,:]/max(temperature,1e-4)

        if repetition_penalty != 1.0:
            used=torch.unique(idx[0,-64:])
            logits[:,used] /= repetition_penalty

        if top_k:
            v,_=torch.topk(logits,min(top_k,logits.size(-1)))
            logits[logits < v[:,-1,None]]=-float("inf")

        probs=F.softmax(logits,dim=-1)

        if top_p < 1.0:
            sorted_probs, sorted_idx=torch.sort(probs,descending=True)
            cumulative=torch.cumsum(sorted_probs,dim=-1)
            mask=cumulative>top_p
            mask[:,1:]=mask[:,:-1].clone()
            mask[:,0]=False
            sorted_probs[mask]=0
            sorted_probs/=sorted_probs.sum(dim=-1,keepdim=True)
            next_sorted=torch.multinomial(sorted_probs,1)
            next_token=sorted_idx.gather(-1,next_sorted)
        else:
            next_token=torch.multinomial(probs,1)

        idx=torch.cat([idx,next_token],dim=1)

    return idx


In [ ]:
def encode(text):
    return tokenizer.encode(text).ids

def decode(ids):
    return tokenizer.decode(ids)

prompt="Once upon a time, a curious little robot"
idx=torch.tensor([encode(prompt)],dtype=torch.long,device=device)
out=generate(model,idx,max_new_tokens=120)
print(decode(out[0].tolist()))


## CRISP-DM 6 — Simple Chatbot

A base TinyStories language model is primarily a text-completion model, so the chatbot uses a lightweight dialogue prompt template.

For stronger instruction-following, the next step would be a short supervised fine-tuning stage on request/response examples.


In [ ]:
def chat(
    user_message,
    temperature=.8,
    top_p=.95,
    top_k=50,
    max_new_tokens=120
):
    prompt=f"User: {user_message}\nAssistant:"
    idx=torch.tensor(
        [encode(prompt)],
        dtype=torch.long,
        device=device
    )
    output=generate(
        model,idx,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )
    text=decode(output[0].tolist())
    return text.split("Assistant:",1)[-1].strip()

print(chat("Tell me a short story about a friendly dragon."))


In [ ]:
# Interactive local chatbot
while True:
    message=input("You: ").strip()
    if message.lower() in {"quit","exit","q"}:
        print("Chat ended.")
        break

    response=chat(message)
    print("MiniLLM:",response)


## Deployment & Monitoring

A lightweight local deployment can wrap `chat()` in Gradio, Streamlit, Flask, or a CLI.

### Monitor
- validation loss / perplexity;
- GPU memory;
- tokens per second;
- repetitive output;
- prompt length;
- failed or empty responses.

### Important limitations
This model is deliberately tiny and trained on a narrow story corpus. It should be treated as an educational LLM architecture project, not a factual general-purpose assistant.


# Final Conclusion

This project builds the core of a modern language model from scratch while keeping the parameter count small enough for laptop experiments.

The important CRISP-DM lesson is that LLM work still requires:

**Business Understanding → Data Understanding → Data Preparation → Modeling → Evaluation → Deployment & Monitoring**

Modern architecture primitives make the model efficient, but dataset quality, evaluation design, and deployment boundaries remain data-science decisions.
